# 02 — Windowing and experiment splits

Reads `cleaned_data/`, slices 2 s windows (50% overlap), and exports one unified window table with experiment-specific split columns.

Experiments:
- **Owner / person-ID**: normal walking only, `session_1` + `session_2` = train, `session_3` = test
- **Context (normal vs crowded)**: for the same participants, use `session_1` = train and `session_2` = test for both normal and crowded; ignore normal `session_3` so the design stays balanced across contexts

Output: `windowed_data/`

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path('.').resolve()
CLEAN = ROOT / 'cleaned_data'
OUT = ROOT / 'windowed_data'
OUT.mkdir(exist_ok=True)

WINDOW_S = 2.0
STEP_S = 1.0   # 50% overlap for 2 s windows
HZ = 100
SENSOR_COLS = ['acc_x', 'acc_y', 'acc_z', 'gyro_x', 'gyro_y', 'gyro_z']

manifest = pd.read_csv(CLEAN / 'manifest.csv')

In [2]:
def extract_windows(df, window_s=WINDOW_S, step_s=STEP_S, hz=HZ):
    win_len = int(window_s * hz)
    step = int(step_s * hz)
    data = df[SENSOR_COLS].to_numpy()
    return np.array([data[i:i + win_len] for i in range(0, len(data) - win_len + 1, step)])


normal_counts = manifest[manifest['context'] == 'normal'].groupby('participant')['session'].nunique()
crowded_counts = manifest[manifest['context'] == 'crowded'].groupby('participant')['session'].nunique()
owner_person_participants = set(normal_counts[normal_counts >= 3].index)
context_participants = owner_person_participants & set(crowded_counts[crowded_counts >= 2].index)


def split_owner_person(participant, context, session):
    if context != 'normal' or participant not in owner_person_participants:
        return 'ignore'
    if session in {'session_1', 'session_2'}:
        return 'train'
    if session == 'session_3':
        return 'test'
    return 'ignore'


def split_context(participant, context, session):
    if participant not in context_participants:
        return 'ignore'
    if session == 'session_1':
        return 'train'
    if session == 'session_2':
        return 'test'
    return 'ignore'


def build_window_tables(records):
    meta_rows = []
    arrays = []
    for _, rec in records.iterrows():
        path = CLEAN / rec['participant'] / rec['context'] / rec['session'] / 'Gait Data.csv'
        X = extract_windows(pd.read_csv(path))
        if len(X) == 0:
            continue
        arrays.append(X)
        split_owner = split_owner_person(rec['participant'], rec['context'], rec['session'])
        split_ctx = split_context(rec['participant'], rec['context'], rec['session'])
        for i in range(len(X)):
            meta_rows.append({
                'participant': rec['participant'],
                'context': rec['context'],
                'session': rec['session'],
                'window_idx': i,
                'label_person': rec['participant'],
                'label_context': rec['context'],
                'split_owner_person': split_owner,
                'split_context': split_ctx,
                'use_owner_person': rec['participant'] in owner_person_participants and rec['context'] == 'normal',
                'use_context': rec['participant'] in context_participants,
            })
    X_all = np.concatenate(arrays, axis=0) if arrays else np.empty((0, int(WINDOW_S * HZ), 6))
    return pd.DataFrame(meta_rows), X_all


meta, X = build_window_tables(manifest)
print('Owner/person participants:', sorted(owner_person_participants))
print('Context participants:', sorted(context_participants))
meta.head()

Owner/person participants: ['Asena', 'Darius', 'Jun', 'Oana', 'Pedro']
Context participants: ['Asena', 'Darius', 'Jun', 'Oana', 'Pedro']


,participant,context,session,window_idx,label_person,label_context,split_owner_person,split_context,use_owner_person,use_context
0,Ana,crowded,session_1,0,Ana,crowded,ignore,ignore,False,False
1,Ana,crowded,session_1,1,Ana,crowded,ignore,ignore,False,False
2,Ana,crowded,session_1,2,Ana,crowded,ignore,ignore,False,False
3,Ana,crowded,session_1,3,Ana,crowded,ignore,ignore,False,False
4,Ana,crowded,session_1,4,Ana,crowded,ignore,ignore,False,False


In [3]:
# save unified window table
np.save(OUT / 'X_all.npy', X)
meta.to_csv(OUT / 'meta_all.csv', index=False)

print('all windows:', X.shape)
print('\nOwner/person split counts:')
print(meta[meta['split_owner_person'] != 'ignore'].groupby(['split_owner_person', 'context']).size())
print('\nContext split counts:')
print(meta[meta['split_context'] != 'ignore'].groupby(['split_context', 'context']).size())

all windows: (4805, 200, 6)

Owner/person split counts:
split_owner_person  context
test                normal      895
train               normal     1790
dtype: int64

Context split counts:
split_context  context
test           crowded    891
               normal     895
train          crowded    887
               normal     895
dtype: int64


In [4]:
# quick summaries for the two experiments
owner_meta = meta[meta['split_owner_person'] != 'ignore']
context_meta = meta[meta['split_context'] != 'ignore']

print('Owner/person experiment participants:', sorted(owner_meta['participant'].unique()))
print(owner_meta.groupby(['split_owner_person', 'participant']).size())
print('\nContext experiment participants:', sorted(context_meta['participant'].unique()))
print(context_meta.groupby(['split_context', 'context']).size())

Owner/person experiment participants: ['Asena', 'Darius', 'Jun', 'Oana', 'Pedro']
split_owner_person  participant
test                Asena          179
                    Darius         179
                    Jun            179
                    Oana           179
                    Pedro          179
train               Asena          358
                    Darius         358
                    Jun            358
                    Oana           358
                    Pedro          358
dtype: int64

Context experiment participants: ['Asena', 'Darius', 'Jun', 'Oana', 'Pedro']
split_context  context
test           crowded    891
               normal     895
train          crowded    887
               normal     895
dtype: int64
